In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from PIL import Image

In [ ]:
# Define augmentation pipeline according to SimCLR paper

transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [ ]:
# Define a wrapper class essentially so augmentation is applied twice -> results in two independent views per image.

class TwoViewDataset:
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, index):
        img = self.base_dataset[index]
        img = Image.fromarray(img) # Ensures image is a PIL image

        view1 = self.transform(img)
        view2 = self.transform(img)

        return (view1, view2)

In [ ]:
# Load base CIFAR10
base_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

# Wrap it -> create trainset and its DataLoader!
trainset = TwoViewDataset(base_dataset=base_cifar10, transform=transform)
trainloader = DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2)